# Mandelbrot & Julia — deep-zoom Dive Studio (CUDA, high precision)

The v2 Studio, rebuilt to go **as deep as you like**: ×10¹⁰⁰⁰ and beyond, on a Colab GPU.

**On Colab:** *Runtime ▸ Change runtime type ▸ A100 GPU* (or L4 / T4), then **run the three cells in order, once**:

1. **Setup** — installs the packages, mounts Google Drive for the films, checks the GPU.
2. **Engine** — the CUDA kernel, the high-precision math, and the movie maker.
3. **Studio** — pick a destination from the menu, look around, preview, and film.

It also runs on a Mac or PC with no GPU — the same kernel, compiled for the CPU cores instead.

**In the Studio**

| You do | It does |
|---|---|
| Pick a *Destination* | flies the viewer there and shows its Julia twin |
| Left-click the picture | zooms toward the click by the *per click* factor (×2 … ×10,000) |
| Right-click / shift-click | zooms back out |
| **🎯 Snap** (shallow views) | finds the exactly self-repeating points nearest your last click |
| **🔭 Find minibrot** (any depth) | finds the minibrot hiding in the view, to as many digits as it needs |
| *Depth 10^* slider | how deep the film goes (up to 10^3010 for the spirals) |
| **👁 Preview** | 12 frames from along the film, plus a time estimate |
| **🎬 Render film** | writes the mp4 in 10-second pieces to Drive (`MyDrive/Mandelbrot_movies`) |

If Colab disconnects in the middle of a film, run the cells again and press **🎬 Render film** with the same settings: it picks up from the last finished piece.

**How it goes so deep**

* **Perturbation.** Only one point — the *reference*, the destination itself — is iterated in high precision (gmpy2, thousands of digits). Every pixel then follows only its tiny *difference* from that orbit, which float64 on the GPU handles fine.
* **Rescaling.** Beyond 10⁻³⁰⁸ the differences are too small even for float64, so they are carried as *number × 2ᵏ* until they grow big enough.
* **Rebasing.** When a pixel's orbit wanders near 0 or near the reference's cycle, it switches to that part of the reference — this keeps every pixel exact (checked against full-precision iteration to 10⁻¹⁰⁰⁰).
* **Strobe.** Spirals repeat every ×|λ| turned by arg λ. Racing down 1,000 decades, each frame jumps a whole number of repeats plus a sliver, so the film looks like one calm, steady zoom while the counter races.
* **Minibrots.** For a minibrot dive the reference is its nucleus (a period-p cycle through 0); pixels inside are recognized when their orbit settles on the cycle.

**What costs time:** iterations, not digits. Spiral dives stay cheap at any depth. Minibrots need about 200 × period iterations at the end (the 10⁻¹⁰⁰⁰ one: ~3 million per pixel), and the two cascade points (airplane, Feigenbaum) multiply their iterations each repeat, so they stop at ×2⁴⁶ and ×2³⁰.

*For A.K. Dewdney's "Computer Recreations," Scientific American, August 1985.*

In [1]:
# 1 — Setup (run once per session)
#   On Colab: Runtime ▸ Change runtime type ▸ A100 GPU first.
%pip install -q gmpy2 numba ipywidgets imageio imageio-ffmpeg pillow

import shutil
import subprocess

SAVE_TO_DRIVE = True  # Colab: change to True for films to go to Google Drive, MyDrive/Mandelbrot_movies

try:                                   # Google Colab only
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and SAVE_TO_DRIVE:
    drive.mount("/content/drive")

if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip())
    from numba import cuda
    if cuda.is_available():
        print("CUDA ready.")
    else:
        %pip install -q "numba-cuda[cu12]"
        print("Installed numba-cuda. Now Runtime ▸ Restart session, then run this cell again.")
else:
    print("No NVIDIA GPU here: the engine will use the CPU cores (fine for exploring; "
          "deep films are slower).")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 126.1 MB/s eta 0:00:00
NVIDIA A100-SXM4-80GB, 81920 MiB
CUDA ready.


In [2]:
# 2 — The deep-zoom engine (run once)
#
# How it goes so deep: PERTURBATION THEORY.
#   * One reference orbit Z0=0, Z1=c, Z2=c²+c, ... is computed with as many
#     digits as the zoom needs (gmpy2 on the CPU -- 3,400 bits for 10^-1000).
#   * Every pixel c+δc only tracks its tiny difference δz from that orbit:
#         δz -> 2·Z·δz + δz² + δc
#     in ordinary float64 -- on the GPU (numba.cuda), or on all CPU cores.
#   * Rescaling: below ~10^-300 float64 underflows, so a pixel's δz is kept as
#     mantissa × 2^k with its own exponent k until it has grown big enough.
#   * Rebasing (Zhuoran, 2021): when a pixel's orbit passes closer to 0 than
#     its δz, it restarts against the start of the reference.  No glitches.
#   * Destinations are refined by Newton's method to thousands of digits.
#
#   compute         escape counts for one view (GPU or CPU, shallow or deep)
#   DESTINATIONS    spirals, branch points, minibrot cascades, deep minibrots
#   minibrot_near   finds the minibrot hiding in any view, at any depth
#   Dive            the zoom from the whole set down to a destination
#   write_movie     renders in 10-second pieces, so a broken run can resume

import math
import os
import time
import hashlib
import shutil
import subprocess
import warnings
from contextlib import contextmanager
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import matplotlib as mpl
import gmpy2
from gmpy2 import mpfr, mpc
import numba
from numba import njit, prange
from PIL import Image, ImageDraw, ImageFont

try:
    from numba.core.errors import NumbaDeprecationWarning, NumbaPerformanceWarning
    warnings.simplefilter("ignore", NumbaDeprecationWarning)
    warnings.simplefilter("ignore", NumbaPerformanceWarning)
except Exception:
    pass
# numba-cuda has its own warning class; the tiny warm-up frame triggers "Grid size 1 ..."
warnings.filterwarnings("ignore", message=r"Grid size \d+ will likely result in GPU under-utilization")
try:
    from numba import cuda
    GPU = bool(cuda.is_available()) and not os.environ.get("MANDEL_FORCE_CPU")
except Exception:
    cuda, GPU = None, False

def device_name():
    if GPU:
        try:
            n = cuda.get_current_device().name
            return "GPU · " + (n.decode() if isinstance(n, bytes) else str(n))
        except Exception:
            return "GPU (CUDA)"
    return f"CPU · {numba.get_num_threads()} threads (no CUDA GPU found)"

SPAN_HOME, CENTER_HOME = 2.9, -0.75 + 0j
BAIL = 1.0e4                 # escape when |z|² > BAIL (|z| > 100): smooth colors
DIRECT_HALVINGS = 36.0       # viewer: shallower than ×2^36, plain float64 will do
DIVE_PERTURB_FROM = 10.0     # dives: perturbation from ×2^10 on (exact near the cycles)
LOG10_2 = math.log10(2.0)
MOVIE_DIR = (Path("/content/drive/MyDrive/Mandelbrot_movies")
             if Path("/content/drive/MyDrive").is_dir() else Path("movies"))

# ------------------------------------------------------- high precision
@contextmanager
def precision(bits):
    ctx = gmpy2.get_context()
    old = ctx.precision
    ctx.precision = max(int(bits), 64)
    try:
        yield
    finally:
        ctx.precision = old

def bits_for(halvings):
    """Bits needed at zoom ×2^halvings (frames up to 2048 px, plus a safety margin)."""
    return int(math.ceil(max(halvings, 0.0) + 11 + 96))

def hp(x, bits):
    """complex / str / mpc  ->  mpc with `bits` of precision."""
    with precision(bits):
        if isinstance(x, tuple):
            return mpc(mpfr(x[0]), mpfr(x[1]))
        return mpc(x)

def _dec(x, n):
    m, e, _ = gmpy2.digits(x, 10, n)
    neg = m.startswith("-")
    m = m.lstrip("-")
    return f"{'-' if neg else ''}0.{m}e{e}"

def hp_str(z, digits=None):
    """(real, imag) as decimal strings with all the digits z holds."""
    n = digits or int(z.precision[0] * LOG10_2) + 2
    return _dec(z.real, n), _dec(z.imag, n)

def short(z, n=16):
    """A readable leading-digits form of a high-precision number."""
    def one(x):
        m, e, _ = gmpy2.digits(x, 10, n + 2)
        neg = m.startswith("-")
        m = m.lstrip("-")
        v = float(f"0.{m}e{e}")
        return f"{'-' if neg else '+'}{abs(v):.{n}f}…"
    return f"{one(z.real)} {one(z.imag)}i"

def ndigits(z):
    return int(z.precision[0] * LOG10_2)

# --------------------------------------------------------- reference orbit
class RefOrbit:
    """Z_n of the high-precision point c, kept as float64 copies for the kernels."""

    def __init__(self, c, bits):
        self.bits = int(bits)
        with precision(self.bits):
            self.c = mpc(c)
            self._z = mpc(0)
        self.Zr = np.zeros(4096)
        self.Zi = np.zeros(4096)
        self.n = 0
        self.escaped = False
        self._dev = None

    @property
    def nref(self):
        return self.n + 1

    def ensure(self, max_iter):
        if self.escaped or self.n >= max_iter:
            return
        if max_iter + 1 > self.Zr.size:
            size = max(max_iter + 1, 2 * self.Zr.size)
            self.Zr = np.concatenate([self.Zr, np.zeros(size - self.Zr.size)])
            self.Zi = np.concatenate([self.Zi, np.zeros(size - self.Zi.size)])
        Zr, Zi, n = self.Zr, self.Zi, self.n
        sq = gmpy2.square
        with precision(self.bits):
            z, c = self._z, self.c
            while n < max_iter:
                z = sq(z) + c
                n += 1
                zc = complex(z)
                Zr[n], Zi[n] = zc.real, zc.imag
                if zc.real * zc.real + zc.imag * zc.imag > BAIL:
                    self.escaped = True
                    break
        self._z, self.n = z, n
        self._dev = None

    def device(self):
        if self._dev is None:
            self._dev = (cuda.to_device(self.Zr[:self.nref].copy()),
                         cuda.to_device(self.Zi[:self.nref].copy()))
        return self._dev

_REFS = {}
def get_ref(c, bits):
    """Reference orbits are reused while the point and precision match."""
    key = (hp_str(c, 40 + int(bits * LOG10_2)), int(bits))
    if key not in _REFS:
        if len(_REFS) >= 6:
            _REFS.pop(next(iter(_REFS)))
        _REFS[key] = RefOrbit(c, bits)
    return _REFS[key]

# ------------------------------------------------------------- the kernel
# One function, compiled twice: for the GPU (numba.cuda) and for the CPU.
_LN2 = 0.6931471805599453
_BIG = 2.0 ** 128
_SMALL = 2.0 ** -128
_TINY = 2.0 ** -900

def _pixel(dr, di, E, direct, cre, cim, Zr, Zi, nref, max_iter, bail, cyc_k, cyc_p):
    """Smooth escape count of the pixel at offset (dr, di)·2^E; inf = inside."""
    if direct:                                   # shallow: plain float64
        u = 2.0 ** E
        cr = cre + dr * u
        ci = cim + di * u
        xq = cr - 0.25
        y2 = ci * ci
        q = xq * xq + y2
        if q * (q + xq) <= 0.25 * y2 or (cr + 1.0) * (cr + 1.0) + y2 <= 0.0625:
            return math.inf                      # main cardioid / period-2 disk
        zr = 0.0
        zi = 0.0
        for k in range(1, max_iter + 1):
            t = zr * zr - zi * zi + cr
            zi = 2.0 * zr * zi + ci
            zr = t
            z2 = zr * zr + zi * zi
            if z2 > bail:
                return k + 1.0 - math.log(math.log(z2) * 0.5 / _LN2) / _LN2
        return math.inf

    it = 0
    n = 0
    dzr = 0.0
    dzi = 0.0
    u = 2.0 ** E                                 # (underflows to 0 when negligible)
    dcr = dr * u
    dci = di * u
    deep = E < -960
    resc = deep                                  # in the rescaled (floatexp) regime?
    pwr = 0.0                                    # nucleus mode: last return near 0,
    pwi = 0.0                                    # = pw·2^pk
    pk = 0
    seen = 0
    k = E
    wr = 0.0
    wi = 0.0
    sk = 1.0
    sc = 1.0
    while it < max_iter:
        if resc:
            # rescaled: δz = w·2^k, δc = d·2^E,   w <- 2Zw + 2^k w² + 2^(E-k) d.
            # Used at the start and every time the orbit comes back very close to 0,
            # where δz² and δc are far below float64's range.
            m0 = max(abs(dzr), abs(dzi))
            if m0 == 0.0:
                k = E
                wr = 0.0
                wi = 0.0
            else:
                k = int(math.floor(math.log2(m0)))
                if k < -1000:
                    k = -1000
                f = 2.0 ** (-k)
                wr = dzr * f
                wi = dzi * f
            sk = 2.0 ** k
            sc = 2.0 ** (E - k)
            while it < max_iter:
                if n == 0:
                    # Z_0 = 0: δz' = δz² + δc. Take the new exponent from the larger
                    # term so neither w² nor d can underflow w to a stuck zero.
                    k2 = 2 * k
                    kn = k2 if k2 > E else E
                    w2r = wr * wr - wi * wi
                    w2i = 2.0 * wr * wi
                    a = 2.0 ** (k2 - kn)
                    b = 2.0 ** (E - kn)
                    wr = w2r * a + dr * b
                    wi = w2i * a + di * b
                    k = kn
                    sk = 2.0 ** k
                    sc = 2.0 ** (E - k)
                else:
                    zr = Zr[n]
                    zi = Zi[n]
                    ar = 2.0 * (zr * wr - zi * wi)
                    ai = 2.0 * (zr * wi + zi * wr)
                    w2r = wr * wr - wi * wi
                    w2i = 2.0 * wr * wi
                    wr = ar + sk * w2r + sc * dr
                    wi = ai + sk * w2i + sc * di
                n += 1
                it += 1
                zr = Zr[n]
                zi = Zi[n]
                z2 = zr * zr + zi * zi           # δz is negligible next to Z here
                if z2 > bail:
                    return it + 1.0 - math.log(math.log(z2) * 0.5 / _LN2) / _LN2
                mag = max(abs(wr), abs(wi))
                if mag > _BIG:
                    wr *= _SMALL
                    wi *= _SMALL
                    k += 128
                    sk = 2.0 ** k
                    sc = 2.0 ** (E - k)
                elif mag < _SMALL and mag > 0.0:
                    wr *= _BIG
                    wi *= _BIG
                    k -= 128
                    sk = 2.0 ** k
                    sc = 2.0 ** (E - k)
                if cyc_k < 0 and n == cyc_p:     # periodic reference: Z_p = Z_0 = 0
                    n = 0
                    # z = δz = w·2^k; inside, these returns converge to a fixed point
                    if seen > 0:
                        sh = pk - k
                        if sh > -1000 and sh < 1000:
                            s = 2.0 ** sh
                            er = wr - pwr * s
                            ei = wi - pwi * s
                            if er * er + ei * ei < 1e-20 * (wr * wr + wi * wi):
                                return math.inf
                    pwr = wr
                    pwi = wi
                    pk = k
                    seen = 1
                if k > -800 or n == nref - 1:
                    break
            if it >= max_iter:
                return math.inf
            f = 2.0 ** k
            dzr = wr * f
            dzi = wi * f
            resc = False
            if n == nref - 1:                    # reference ran out: rebase
                dzr += Zr[n]
                dzi += Zi[n]
                n = 0
                resc = deep and dzr * dzr + dzi * dzi < _TINY
            continue

        # float64 perturbation + rebasing
        zr = Zr[n]
        zi = Zi[n]
        tr = 2.0 * zr + dzr
        ti = 2.0 * zi + dzi
        t = tr * dzr - ti * dzi + dcr
        dzi = tr * dzi + ti * dzr + dci
        dzr = t
        n += 1
        it += 1
        zr = Zr[n] + dzr
        zi = Zi[n] + dzi
        z2 = zr * zr + zi * zi
        if z2 > bail:
            return it + 1.0 - math.log(math.log(z2) * 0.5 / _LN2) / _LN2
        d2 = dzr * dzr + dzi * dzi
        if z2 < d2 or n == nref - 1 or (cyc_k < 0 and n == cyc_p):
            dzr = zr
            dzi = zi
            n = 0
            if cyc_k < 0:
                # inside the minibrot the orbit settles on an attracting cycle:
                # its returns near 0 stop moving (compared relative to their size)
                s = 2.0 ** pk
                er = zr - pwr * s
                ei = zi - pwi * s
                if seen > 0 and er * er + ei * ei < 1e-20 * z2:
                    return math.inf
                pwr = zr
                pwi = zi
                pk = 0
                seen = 1
            resc = deep and z2 < _TINY           # too close to 0 for float64: rescale
        elif cyc_k >= 0 and cyc_p > 0 and d2 > 1e-8:
            # the reference ends on a repelling cycle: when the pixel comes closer
            # to another point of that cycle, continue from there (full precision)
            best = 0.25 * d2
            bj = -1
            for j in range(cyc_k, cyc_k + cyc_p):
                er = zr - Zr[j]
                ei = zi - Zi[j]
                e2 = er * er + ei * ei
                if e2 < best:
                    best = e2
                    bj = j
            if bj >= 0:
                dzr = zr - Zr[bj]
                dzi = zi - Zi[bj]
                n = bj
    return math.inf

_FAST = {"nsz", "arcp", "contract", "afn", "reassoc"}
_pixel_cpu = njit(fastmath=_FAST)(_pixel)

@njit(parallel=True, fastmath=_FAST)
def _render_cpu(W, H, m, offr, offi, E, direct, cre, cim, Zr, Zi, nref, max_iter, bail,
                cyc_k, cyc_p):
    out = np.empty((H, W))
    for j in prange(H):
        di = (j - 0.5 * (H - 1)) * m + offi
        for i in range(W):
            dr = (i - 0.5 * (W - 1)) * m + offr
            out[j, i] = _pixel_cpu(dr, di, E, direct, cre, cim, Zr, Zi, nref, max_iter, bail,
                                   cyc_k, cyc_p)
    return out

if GPU:
    _pixel_gpu = cuda.jit(device=True)(_pixel)

    @cuda.jit
    def _render_gpu(out, W, H, m, offr, offi, E, direct, cre, cim, Zr, Zi, nref, max_iter,
                    bail, cyc_k, cyc_p):
        i, j = cuda.grid(2)
        if i < W and j < H:
            dr = (i - 0.5 * (W - 1)) * m + offr
            di = (j - 0.5 * (H - 1)) * m + offi
            out[j, i] = _pixel_gpu(dr, di, E, direct, cre, cim, Zr, Zi, nref, max_iter, bail,
                                   cyc_k, cyc_p)

_DUMMY = np.zeros(2)
_DUMMY_DEV = []

def grid(depth, px):
    """Pixel spacing m·2^E (1 <= m < 2) of a px-wide view at zoom ×2^depth."""
    l = math.log2(SPAN_HOME / max(px - 1, 1)) - depth
    E = math.floor(l)
    return 2.0 ** (l - E), int(E)

def compute(depth, px, max_iter, center=CENTER_HOME, ref=None, off=0j, cycle=(0, 0)):
    """Smooth escape counts of a px × px view at zoom ×2^depth (row 0 = bottom).
       ref=None: plain float64 around `center` (fine for shallow views).
       Otherwise perturbation against `ref`; off = (view center − ref.c) / 2^E;
       cycle = (k, p) if ref's orbit lands on a p-cycle at step k."""
    m, E = grid(depth, px)
    max_iter = int(max(max_iter, 16))
    direct = ref is None
    cyc_k, cyc_p = (int(cycle[0]), int(cycle[1])) if not direct else (0, 0)
    if direct:
        Zr = Zi = _DUMMY
        nref, offr, offi = 2, 0.0, 0.0
        cre, cim = float(center.real), float(center.imag)
    else:
        ref.ensure(max(max_iter, cyc_k + cyc_p + 1))
        Zr, Zi, nref = ref.Zr, ref.Zi, ref.nref
        offr, offi, cre, cim = float(off.real), float(off.imag), 0.0, 0.0
        if cyc_k + cyc_p >= nref:
            cyc_p = 0
    if not GPU:
        return _render_cpu(px, px, m, offr, offi, E, direct, cre, cim, Zr, Zi, nref,
                           max_iter, BAIL, cyc_k, cyc_p)
    if direct:
        if not _DUMMY_DEV:
            _DUMMY_DEV.extend([cuda.to_device(_DUMMY), cuda.to_device(_DUMMY)])
        dZr, dZi = _DUMMY_DEV
    else:
        dZr, dZi = ref.device()
    out = cuda.device_array((px, px), dtype=np.float64)
    tpb = (16, 16)
    bpg = ((px + 15) // 16, (px + 15) // 16)
    _render_gpu[bpg, tpb](out, px, px, m, offr, offi, E, direct, cre, cim, dZr, dZi,
                          nref, max_iter, BAIL, cyc_k, cyc_p)
    return out.copy_to_host()

def auto_iter(depth):
    return int(min(300 + 60 * min(depth, 40.0) + 30 * max(depth - 40.0, 0.0), 2_000_000))

# ---------------------------------------------------------------- Julia twin
@njit(parallel=True, fastmath=_FAST)
def _julia_counts(re, im, max_iter, cre, cim):
    out = np.empty((im.size, re.size))
    R = 0.5 * (1.0 + np.sqrt(1.0 + 4.0 * np.hypot(cre, cim)))
    bail = 16.0 if R < 4.0 else R * R
    for j in prange(im.size):
        for i in range(re.size):
            zr, zi = re[i], im[j]
            mu = np.inf
            for k in range(1, max_iter + 1):
                zr, zi = zr * zr - zi * zi + cre, 2.0 * zr * zi + cim
                z2 = zr * zr + zi * zi
                if z2 > bail:
                    mu = k - np.log2(np.log2(z2) * 0.5) + 1.0
                    break
            out[j, i] = mu
    return out

def julia(c, px, max_iter=600):
    span = max(3.2, 2.6 * (0.5 + math.sqrt(0.25 + abs(c))))
    ax = np.linspace(-span / 2, span / 2, px)
    return _julia_counts(ax, ax, max_iter, c.real, c.imag)

def colorize(mu, cmap="magma", lim=None):
    """mu -> RGB uint8 (row 0 = top).  lim=None: central 99% of escaped pixels."""
    finite = mu[np.isfinite(mu)]
    if lim is None:
        lo, hi = np.percentile(finite, [0.5, 99.5]) if finite.size else (0.0, 1.0)
    else:
        lo, hi = lim
    v = np.clip((mu - lo) / max(hi - lo, 1e-12), 0.0, 1.0)
    rgba = mpl.colormaps[cmap](np.nan_to_num(v), bytes=True)
    rgba[~np.isfinite(mu)] = (0, 0, 0, 255)
    return np.ascontiguousarray(rgba[::-1, :, :3])

# ------------------------------------------------- self-repeating points
# A Misiurewicz point: the critical orbit lands after k steps (preperiod) on a
# repelling cycle of period p.  Around it the set repeats every |λ| of zoom,
# turned by arg λ, where λ = the product of 2z over the cycle.

def _newton_mis(c, k, p, steps=80):
    step = 1.0
    for _ in range(steps):
        z = dz = zk = dzk = 0j
        for n in range(1, k + p + 1):
            dz = 2.0 * z * dz + 1.0
            z = z * z + c
            if n == k:
                zk, dzk = z, dz
            if abs(z) > 1e6:
                return None
        if dz == dzk:
            return None
        step = (z - zk) / (dz - dzk)
        c -= step
        if abs(step) < 1e-16 * max(1.0, abs(c)):
            return c
    return c if abs(step) < 1e-12 else None

def classify(c, K=40, P=16):
    """(preperiod k, period p, λ) of c's critical orbit, or None."""
    zs = [0j]
    for _ in range(K + 2 * P + 2):
        zs.append(zs[-1] ** 2 + c)
    for p in range(1, P + 1):
        for k in range(0, K + 1):
            tol = 1e-9 * (1.0 + abs(zs[k]))
            if abs(zs[k + p] - zs[k]) < tol:
                if abs(zs[k + 2 * p] - zs[k + p]) > tol or k == 0:
                    return None
                lam = 1.0 + 0j
                for j in range(k, k + p):
                    lam *= 2.0 * zs[j]
                return (k, p, lam) if abs(lam) > 1.0 + 1e-9 else None
    return None

def snap_misiurewicz(c0, radius=None, K=24, P=8, top=6):
    """Self-repeating points near c0 (float64 -- for shallow views), nearest first."""
    found = []
    for p in range(1, P + 1):
        for k in range(2, K + 1):
            c = _newton_mis(complex(c0), k, p)
            if c is None or not (np.isfinite(c.real) and np.isfinite(c.imag)):
                continue
            info = classify(c)
            if info is None or (radius is not None and abs(c - c0) > radius):
                continue
            if any(abs(c - f[0]) < 1e-11 * max(1.0, abs(c)) for f in found):
                continue
            found.append((c, *info))
    found.sort(key=lambda f: abs(f[0] - c0))
    return found[:top]

def refine_misiurewicz(c, k, p, bits):
    """Newton's method on z_(k+p)(c) = z_k(c), carried to `bits` of precision."""
    with precision(bits + 32):
        C = mpc(c)
        tiny = mpfr(2) ** (-(bits + 8))
        for _ in range(64):
            z = dz = zk = dzk = mpc(0)
            for n in range(1, k + p + 1):
                dz = 2 * z * dz + 1
                z = z * z + C
                if n == k:
                    zk, dzk = z, dz
            step = (z - zk) / (dz - dzk)
            C -= step
            if abs(step) <= tiny * (1 + abs(C)):
                break
    return hp(C, bits)

# ------------------------------------------------------ minibrots (any depth)
def find_period(center, radius, max_period, bits):
    """Lowest period of a minibrot nucleus inside the disk |c - center| < radius
       (ball arithmetic: the first n where the image of the disk contains 0)."""
    with precision(bits):
        c = mpc(center)
        r = mpfr(radius)
        z = mpc(0)
        R = mpfr(0)
        for n in range(1, max_period + 1):
            R = (2 * abs(z) + R) * R + r
            z = gmpy2.square(z) + c
            az = abs(z)
            if az < R:
                return n
            if az > 1e3:
                return None
    return None

def find_nucleus(c0, p, bits, steps=100):
    """Newton's method on z_p(c) = 0 from c0."""
    with precision(bits + 32):
        c = mpc(c0)
        tiny = mpfr(2) ** (-(bits + 8))
        for _ in range(steps):
            z = dz = mpc(0)
            for _ in range(p):
                dz = 2 * z * dz + 1
                z = gmpy2.square(z) + c
            if dz == 0:
                return None
            step = z / dz
            c -= step
            if abs(step) <= tiny * (1 + abs(c)):
                return hp(c, bits)
    return None

def atom_size(nuc, p, bits):
    """Size and orientation of the period-p minibrot at nucleus nuc, relative to
       the whole set: it looks like  nuc + size·(Mandelbrot set).  -> mpc"""
    with precision(bits + 32):
        c = mpc(nuc)
        z = mpc(0)
        l = mpc(1)
        b = mpc(1)
        for _ in range(1, p):
            z = gmpy2.square(z) + c
            l = 2 * z * l
            b = b + 1 / l
        return 1 / (b * l * l)

def minibrot_near(center, depth, max_period=400_000, status=None):
    """The lowest-period minibrot in the view (center, zoom ×2^depth) -> (nuc, p, size)."""
    bits = bits_for(depth + 4)
    radius = hp(SPAN_HOME / 2 * math.sqrt(2.0), bits).real * mpfr(2) ** (-depth)
    p = find_period(center, radius, max_period, bits)
    if p is None:
        return None
    if status:
        status(f"period {p:,d} — Newton's method to {int(bits * LOG10_2):,d} digits …")
    nuc = find_nucleus(center, p, bits)
    if nuc is None:
        return None
    size = atom_size(nuc, p, bits)
    if abs(size) == 0:
        return None
    # carry the nucleus deep enough to frame the minibrot itself
    need = bits_for(-float(gmpy2.log2(abs(size))))
    if need > bits:
        nuc = find_nucleus(nuc, p, need)
        if nuc is None:
            return None
        size = atom_size(nuc, p, need)
    return nuc, p, size

# ----------------------------------------------------------- destinations
@dataclass
class Dest:
    """kind = "misiurewicz" (spirals/branches, repeat every |λ| turned by arg λ),
       "renorm" (minibrots inside minibrots, ×δ per repeat) or
       "minibrot" (a zoom that ends on one deep minibrot)."""
    name: str
    c: complex
    kind: str
    scale: float = 2.0           # zoom factor between repeats
    turn: float = 0.0            # degrees turned per repeat
    p: int = 1
    k: int = 0
    lam: complex = 0j
    ratio: int = 1
    anchor: complex = None       # opening-frame center (None -> c)
    max_halvings: float = 10_000.0
    blurb: str = ""
    c_str: tuple = None          # minibrot: nucleus as decimal strings
    size_str: tuple = None       # minibrot: size·e^(i·angle) as decimal strings

    def __post_init__(self):
        self._hp = {}
        if self.kind == "minibrot":
            s = hp(self.size_str, 64)
            self.size_log2 = float(gmpy2.log2(abs(s)))
            e = math.floor(self.size_log2)
            self.size_m = complex(s * mpfr(2) ** (-e))   # size = size_m · 2^size_e
            self.size_e = e
            self.angle = math.degrees(math.atan2(self.size_m.imag, self.size_m.real))
            self.max_halvings = max(-self.size_log2, 1.0)
            self.anchor = CENTER_HOME

    @classmethod
    def misiurewicz(cls, name, c, blurb=""):
        info = classify(c)
        if info is None:
            raise ValueError(f"{name}: {c} is not a Misiurewicz point")
        k, p, lam = info
        return cls(name, c, "misiurewicz", abs(lam), math.degrees(np.angle(lam)),
                   p=p, k=k, lam=lam, blurb=blurb)

    @classmethod
    def minibrot(cls, name, nuc, p, size, blurb=""):
        digits = ndigits(nuc)
        return cls(name, complex(nuc), "minibrot", p=p, blurb=blurb,
                   c_str=hp_str(nuc, digits), size_str=hp_str(size, 20))

    @property
    def log2_scale(self):
        return math.log2(self.scale) if self.kind != "minibrot" else 1.0

    @property
    def can_turn(self):
        return (self.kind == "misiurewicz" and abs(self.turn) > 1e-9) or \
               (self.kind == "minibrot" and abs(self.angle) > 0.5)

    def levels(self, halvings):
        return halvings / self.log2_scale

    def ref_hp(self, bits):
        """The point itself, with `bits` of precision (Newton-refined)."""
        b = max(64, int(bits))
        if b not in self._hp:
            if self.kind == "misiurewicz":
                self._hp[b] = refine_misiurewicz(self.c, self.k, self.p, b)
            elif self.kind == "renorm":
                self._hp[b] = hp(self.c, b)
            else:
                z = hp(self.c_str, b)
                if b > len(self.c_str[0]) * 3.32 - 16:
                    z = find_nucleus(z, self.p, b) or z
                self._hp[b] = z
        return self._hp[b]

    def summary(self):
        if self.kind == "misiurewicz":
            s = (f"Misiurewicz point · preperiod {self.k}, period {self.p} · "
                 f"λ = {self.lam.real:+.4g}{self.lam.imag:+.4g}i · repeats every "
                 f"×{self.scale:.4g}, turning {self.turn:+.1f}°")
        elif self.kind == "renorm":
            s = (f"Renormalization point · periods ×{self.ratio} per level · "
                 f"repeats every ×{self.scale:.5g}, no turn")
        else:
            s = (f"Minibrot of period {self.p:,d} · nucleus known to {len(self.c_str[0]) - 4:,d} "
                 f"digits · tilted {self.angle:+.0f}°")
        return s + f" · deepest ×10^{self.max_halvings * LOG10_2:,.0f}"

# Deep minibrots, found with minibrot_near() and stored with all their digits.
_DEEP = [
    # (name, period, nucleus re, nucleus im, size re, size im, blurb)
    ('Deep minibrot in Seahorse Valley — ×10^120', 1841,
     "-0.743291890852430202931624325972510757176348558120776581832344743448837354398215431"
     "86328994528251179326814685414633418478988941655278314028425681040418445e0",
     "0.1312405523087976047708459065814781430771141511585840067313129806842480354197856929"
     "9512227401712755843519727545078907005393424960308527423784346306440022e0",
     '0.79488741469381103328e-120', '-0.31057535567276942081e-120',
     "A perfect little copy of the whole set, 10^-120 across, hidden in the seahorse spira"
     "l. The film falls through spiral after spiral, then the minibrot opens up and turns "
     "upright."),
    ('Deep minibrot under the three-armed star — ×10^299', 1209,
     "-0.101096363845622161025785445738622565463805442826253483876931177660780840740470584"
     "274821219810516779033404531908556741193971546144260911882355703907679690981960947477"
     "443575986771264458666304996998920265703168301153130170735692163251780715726135740479"
     "1631979657233986840287113131576840015391010973016334174607652599820561452455838342e0",
     "0.9562865108091415007710960577299774358098333365105291700343143215005246590657167325"
     "269784107873398072043444724926469284366752406567465722656200815719768894007670062514"
     "836059511087263072622809211291924583328093364121346337666353326855255803685326650173"
     "239413496675859918283359372402598580653719452739603679919506074721113760095946093e0",
     '-0.39503139773391080363e-299', '-0.10733522973430854746e-298',
     "Below the branch point M(4,1): stars with 3, 6, 12, 24 … arms nest inside each other"
     " for 300 decades, down to a period-1,209 minibrot."),
    ('Deep minibrot in Elephant Valley — ×10^301', 2900,
     "0.2848845372731743495313019273356873013459617541575097207359516104324498527302635829"
     "563770446702118742304631428297974381997883042405790543266165423575774235985429829124"
     "717217213048219984340760478769110425606724224350555328779402320438245836241582469694"
     "9028252364915405821176772751405434407715315738153171002916379929705617543363957808e0",
     "0.1112182188958193229914056176252321718149178475565732000889208199581909657039084016"
     "998108660912170622692175099955738272789131467806051023663181155845079034025575413941"
     "097343697127283736896816904833436713433646519048874513616997162330938318774616596556"
     "9411121355713499640898476543663651862951142108489368536193936808039167113867302881e-"
     "1",
     '-0.27337691638021580304e-300', '0.86173125324979781393e-301',
     "Down the elephant's trunk spiral for 300 decades to a period-2,900 minibrot."),
    ('Seahorse abyss — a minibrot at ×10^1000', 16037,
     "-0.743291890852430202931624325972510757176348558120776581832330454160928347257478597"
     "718653257133197437231201536472588305583929644073689764431555478587836718290861127025"
     "145634909347572378244778589787013649966685461677089362429706374182778141527564435372"
     "568872765057908790873751472045311204154620446293597205739518368893466359658568361842"
     "301600491483401752216761042137295808200557229907433272846068061599679367781208313553"
     "052184741862893966960988284528580383495018745329333290145004749058776373990727900267"
     "147031401782604811018898243565939938677425106247532405752040276407218869754522373972"
     "366954014320959254773920420378533550026882122467146628952991912022140024858631763841"
     "377660850457304013775195002958339580653276404592601108665141824842063433086093610993"
     "337716671570596166362019734002531870014502845532643434706010921293185305892942270887"
     "545338987870285275323024521214923169225564488551301975751388463385927024344943867697"
     "922186772965641672584197200721951223416224626882247041364050727607964009211118558910"
     "16581680242720968659078937e0",
     "0.1312405523087976047708459065814781430771141511585840067313339426441224485804463828"
     "704809547062528759706770520637320224159702666378411381293727697380576634323825795329"
     "443433122163779881244894544298238870860082516466489520041039964920179255382099101070"
     "060458043609646389324737321675437633675972715936061682899264387492378905313385957335"
     "812736883821230405915647568049753669899839184567394739592781063894258426644572074255"
     "213583943446289132458765130690145155930250901211536848973893702093949741410872538990"
     "791853542922574470127536125009765713340721576727498423867240054491020807685192839645"
     "195761980130671145931852355095746991356437421499987533465423693654341154634326816899"
     "371735442101589400311636668653543228123579355873820738395380588113606723414747035982"
     "055854779080854574589474305855655994757304592794557135045278204495530114856894761175"
     "628264723947469373471149531554260318243031120328763283595243024375289400643727343151"
     "998784672091159884492604994049958195940913642042291384236728252507713254272008573262"
     "4986425073594997511330984e0",
     '0.86345480180718584860e-999', '-0.26430214504119417421e-998',
     "The deep end: 1,000 decades down Seahorse Valley to a period-16,037 minibrot (its ce"
     "nter needs 1,031 digits). The last frames need ~3 million iterations per pixel — sec"
     "onds per frame on an A100, close to a minute on a CPU."),
]

DESTINATIONS = {d.name: d for d in [
    Dest.misiurewicz("Seahorse Valley — the endless spiral",
                     -0.743291890852430 + 0.131240552308798j,
                     "In the valley between the main cardioid and the big period-2 disk. "
                     "It grows only 1.15× per repeat, so the spiral turns and turns."),
    Dest.misiurewicz("Seahorse Valley — double-spiral star",
                     -0.747188946944443 + 0.113383020648328j,
                     "A little lower in the same valley: a starburst of paired spirals, "
                     "a quarter-turn per repeat."),
    Dest.misiurewicz("Elephant Valley — trunk spiral",
                     0.284884537273174 + 0.011121821889582j,
                     "Right of the cardioid, where the 'elephants' march in toward the cusp at c = 1/4."),
    Dest.misiurewicz("Southern spiral valley",
                     -0.774672446935674 - 0.137429292340917j,
                     "Lower rim of Seahorse Valley. λ ≈ 3 with almost no twist — the spirals come straight at you."),
    Dest.misiurewicz("Three-armed branch point M(4,1)",
                     -0.101096363845622 + 0.956286510809142j,
                     "Top of the period-3 bulb, where three filaments meet; each repeat turns ~120°, "
                     "and six-armed stars bloom along the way."),
    Dest.misiurewicz("Upper dendrite fork",
                     -0.175890705973462 + 1.086624831861379j,
                     "Where the top antenna forks; the branches repeat with a 123° twist."),
    Dest.misiurewicz("c = i — Dewdney's dendrite tip",
                     0.0 + 1.0j,
                     "The 1985 classic: 0 → i → −1+i → −i → −1+i → … so λ = 4+4i, 45° per repeat."),
    Dest.misiurewicz("Antenna crossroads −1.8393",
                     -1.839286755214161 + 0.0j,
                     "On the real antenna, left of the airplane minibrot. λ ≈ −3.36: "
                     "every repeat flips the picture end for end."),
    Dest("Airplane cascade — minibrots forever (tripling point)",
         -1.78644025556364 + 0j, "renorm", 55.247, ratio=3, anchor=-0.75 + 0j, max_halvings=46,
         blurb="The period-3 'airplane' minibrot holds its own airplane (period 9), which holds "
               "another (27) … A full copy of the set returns every ×55.247. Escape counts triple "
               "each repeat, so this one is limited by time, not by digits."),
    Dest("Feigenbaum point — period doubling forever",
         -1.4011551890920506 + 0j, "renorm", 4.669201609, ratio=2, max_halvings=30,
         blurb="The end of the bulb chain 2, 4, 8, 16, … on the real axis. Repeats every "
               "×4.6692 (Feigenbaum's constant). Escape counts double each repeat, so it "
               "stops at ×2^30."),
] + [Dest.minibrot(n, hp((re, im), int(len(re) * 3.33) + 64), p, hp((sr, si), 64), b)
     for n, p, re, im, sr, si, b in _DEEP]}

# ---------------------------------------------------------------- the dive
_fontdir = Path(mpl.get_data_path()) / "fonts" / "ttf"
_font_cache = {}
def _font(name, size):
    key = (name, max(int(size), 6))
    if key not in _font_cache:
        try:
            _font_cache[key] = ImageFont.truetype(str(_fontdir / name), key[1])
        except Exception:
            _font_cache[key] = ImageFont.load_default()
    return _font_cache[key]

def _fit(dr, text, name, size, width):
    while size > 7 and dr.textlength(text, font=_font(name, size)) > width:
        size -= 1
    return _font(name, size)
_RS = getattr(Image, "Resampling", Image)
_SUP = str.maketrans("0123456789-", "⁰¹²³⁴⁵⁶⁷⁸⁹⁻")

def zoom_text(depth):
    if depth < 40:
        return f"× {round(2.0 ** depth):,d}"
    e10 = depth * LOG10_2
    ex = math.floor(e10)
    return f"× {10 ** (e10 - ex):.2f} × 10{str(ex).translate(_SUP)}"

class Dive:
    """Constant-speed zoom (in log2) from the whole set down to dest."""

    def __init__(self, dest, halvings=None, cmap="magma", corotate=False, strobe=True,
                 detail=1.0, frames_per_repeat=90):
        self.d = dest
        self.halvings = min(halvings or dest.max_halvings, dest.max_halvings)
        self.cmap = cmap
        self.detail = detail
        self.strobe = strobe and dest.kind == "misiurewicz"
        self.rot = (corotate or self.strobe) and dest.can_turn
        self.fpr = frames_per_repeat
        self.bits = bits_for(self.halvings)
        self.lim = None
        self._ref = None

    # -- where the camera is
    def ref(self):
        if self._ref is None:
            self._ref = get_ref(self.d.ref_hp(self.bits), self.bits)
        return self._ref

    def _center(self, depth, E):
        """View center as float64 (shallow) and as an offset from the reference
           point in units of 2^E (deep)."""
        d = self.d
        a = d.anchor if d.anchor is not None else d.c
        fade = (1.0 - depth / self.d.max_halvings) if d.kind == "minibrot" else 1.0
        center = d.c + (a - d.c) * (2.0 ** -depth) * fade
        off = (a - d.c) * fade * 2.0 ** (-depth - E)
        if d.kind == "minibrot":     # the zoom ends centered on the minibrot, not its nucleus
            off += -0.75 * d.size_m * 2.0 ** (d.size_e - E)
        return center, off

    def max_iter(self, depth):
        d = self.d
        base = 300 + 60 * min(depth, 40.0)
        if d.kind == "renorm":
            n = min(300 * d.ratio ** (depth / d.log2_scale), 4_000_000 if GPU else 650_000)
        elif d.kind == "misiurewicz":
            n = base + d.p * depth / d.log2_scale
        else:
            f = depth / d.max_halvings
            n = max(base + 8 * max(depth - 40.0, 0.0), 200 * d.p * f ** 3)
        return int(n * self.detail)

    def angle(self, depth):
        if not self.rot:
            return 0.0
        if self.d.kind == "minibrot":           # ends with the minibrot upright
            return -self.d.angle * depth / self.d.max_halvings
        return self.d.turn * depth / self.d.log2_scale

    def schedule(self, N):
        """Depth of each frame.  Strobe: once the repeats have settled in, every
           frame jumps a whole number of repeats plus a sliver, so even a race to
           10^-1000 plays as a calm, steady zoom."""
        H = self.halvings
        lin = np.linspace(0.0, H, N)
        ls = self.d.log2_scale
        if not self.strobe or N < 10 or H / ls / (N - 1) <= 0.5:
            return lin
        A = int(max(2, min(N // 5, 120)))
        D_a = min(H, max(3 * ls, 30.0))
        L_a, L_tot = D_a / ls, H / ls
        eps = 1.0 / self.fpr
        per = (L_tot - L_a) / (N - 1 - A)
        if per < 1.0 + eps:
            return lin
        step = math.floor(per - eps) + eps
        head = np.linspace(0.0, D_a, A + 1)
        tail = (L_a + step * np.arange(1, N - A)) * ls
        return np.concatenate([head, tail])

    # -- pictures
    def raw(self, depth, px):
        dd = depth
        if self.rot:                            # bigger square, rotated and cropped later
            px, dd = int(math.ceil(px * math.sqrt(2.0))), depth - 0.5
        m, E = grid(dd, px)
        center, off = self._center(depth, E)
        if dd < DIVE_PERTURB_FROM:
            return compute(dd, px, self.max_iter(depth), center)
        d = self.d
        cycle = ((d.k, d.p) if d.kind == "misiurewicz" else
                 (-1, d.p) if d.kind == "minibrot" else (0, 0))
        return compute(dd, px, self.max_iter(depth), center, self.ref(), off, cycle)

    def _shift(self, mu, depth):
        level = depth / self.d.log2_scale
        if self.d.kind == "renorm":
            x = np.full(mu.shape, np.inf)
            ok = np.isfinite(mu) & (mu > 0)
            x[ok] = np.log(mu[ok] / self.d.ratio ** level)
            return x
        if self.d.kind == "minibrot":
            return np.log(np.maximum(mu, 1.0))
        return mu - self.d.p * level

    def calibrate(self, px=240):
        if self.d.kind == "minibrot":
            self.lim = "auto"
            return
        depth = min(self.halvings, max(2.0 * self.d.log2_scale, 12.0))
        x = self._shift(self.raw(depth, px), depth)
        f = x[np.isfinite(x)]
        lo, hi = np.percentile(f, [0.5, 99.5]) if f.size else (0.0, 1.0)
        self.lim = (lo, hi if hi > lo else lo + 1.0)

    def picture(self, depth, px):
        if self.lim is None:
            self.calibrate()
        x = self._shift(self.raw(depth, px), depth)
        rgb = colorize(x, self.cmap, None if self.lim == "auto" else self.lim)
        img = Image.fromarray(rgb)
        if self.rot:
            img = img.rotate(self.angle(depth), resample=_RS.BICUBIC)
            o = (img.width - px) // 2
            img = img.crop((o, o, o + px, o + px))
        return img

    def frame(self, kf, N, px=600, depth=None):
        """Frame kf of N with its titles."""
        if depth is None:
            depth = self.schedule(N)[kf]
        d = self.d
        level = depth / d.log2_scale
        u = px / 600.0
        margin = int(round(120 * u))
        S = px + 2 * margin
        canvas = Image.new("RGB", (S, S), (0, 0, 0))
        canvas.paste(self.picture(depth, px), (margin, margin))
        dr = ImageDraw.Draw(canvas)
        dr.rectangle([margin - 1, margin - 1, margin + px, margin + px], outline=(60, 60, 60))
        y0 = int(margin * 0.22)
        title = d.name.upper()
        dr.text((margin, y0), title, font=_fit(dr, title, "DejaVuSans.ttf", 18 * u, px),
                fill=(215, 215, 215))
        f2 = _font("DejaVuSans.ttf", 15 * u)
        z = d.ref_hp(self.bits)
        dr.text((margin, y0 + 26 * u), f"c* = {short(z, 13)}  ({ndigits(z):,d} digits)",
                font=f2, fill=(160, 160, 160))
        tag = (f"|λ| = {d.scale:.3g} · {d.turn:+.1f}°" if d.kind == "misiurewicz" else
               f"δ = {d.scale:.5g}" if d.kind == "renorm" else f"period {d.p:,d}")
        dr.text((margin + px - dr.textlength(tag, font=f2), y0 + 48 * u), tag,
                font=f2, fill=(215, 215, 215))
        big = zoom_text(depth)
        fb = _fit(dr, big, "DejaVuSans-Bold.ttf", 46 * u, S - 20)
        dr.text(((S - dr.textlength(big, font=fb)) / 2, margin + px + 16 * u), big,
                font=fb, fill=(255, 210, 74))
        if d.kind == "renorm":
            n = int(round(level))
            what = "the Mandelbrot set" if n == 0 else f"level {n} · period ×{d.ratio}^{n}"
            sub = f"repeat {level:5.2f} / {d.levels(self.halvings):.1f} · {what}"
        elif d.kind == "misiurewicz":
            what = "camera turning with the spiral" if self.rot else f"turned {d.turn * level:+,.0f}°"
            sub = f"repeat {level:,.2f} / {d.levels(self.halvings):,.0f} · {what}"
        else:
            sub = f"{100 * depth / d.max_halvings:.0f}% of the way down to the minibrot"
        sub += f" · frame {kf + 1:,d}/{N:,d}"
        fs = _fit(dr, sub, "DejaVuSans.ttf", 16 * u, S - 20)
        dr.text(((S - dr.textlength(sub, font=fs)) / 2, margin + px + 80 * u), sub,
                font=fs, fill=(150, 150, 150))
        return canvas

    def prepare(self, N):
        """Reference orbit + colors, done once before rendering."""
        if self.halvings >= DIVE_PERTURB_FROM - 0.5:
            depths = self.schedule(N)
            self.ref().ensure(max(self.max_iter(x) for x in depths))
        if self.lim is None:
            self.calibrate()

def contact_sheet(dive, N, px=600, tiles=12, inner=220):
    """12 frames spread along the film.  Also returns an estimate of seconds per frame."""
    depths = dive.schedule(N)
    picks = np.linspace(0, N - 1, tiles).round().astype(int)
    cols, gap = 3, 10
    rows = (tiles + cols - 1) // cols
    tile = inner + 2 * int(round(120 * inner / 600.0))
    sheet = Image.new("RGB", (cols * tile + (cols + 1) * gap, rows * tile + (rows + 1) * gap),
                      (12, 12, 12))
    dive.prepare(N)
    t_total = 0.0
    for t, kf in enumerate(picks):
        t0 = time.time()
        img = dive.frame(int(kf), N, px=inner, depth=depths[kf])
        t_total += time.time() - t0
        sheet.paste(img, (gap + (t % cols) * (tile + gap), gap + (t // cols) * (tile + gap)))
    per_frame = t_total / tiles * (px / inner) ** 2 + 0.04
    return sheet, per_frame

def film_path(dive, N, fps, px):
    """Same settings -> same file name, so an interrupted film resumes."""
    d = dive.d
    key = repr((d.name, d.c_str or d.c, round(dive.halvings, 6), N, fps, px, dive.cmap,
                dive.rot, dive.strobe, dive.detail))
    tag = hashlib.sha1(key.encode()).hexdigest()[:6]
    slug = "".join(ch if ch.isalnum() else "_" for ch in d.name.split("—")[0].strip().lower())
    slug = "_".join(filter(None, slug.split("_")))[:36] or "dive"
    return MOVIE_DIR / f"{slug}_1e-{dive.halvings * LOG10_2:.0f}_{px}px_{tag}.mp4"

def write_movie(dive, path, N, fps=30, px=600, progress=None, piece_seconds=10):
    """Renders in pieces of piece_seconds; finished pieces are kept, so running
       it again with the same settings picks up where it stopped."""
    import imageio.v2 as imageio
    import imageio_ffmpeg
    path = Path(path)
    parts = path.parent / (path.stem + "_parts")
    parts.mkdir(parents=True, exist_ok=True)
    depths = dive.schedule(N)
    dive.prepare(N)
    per = int(fps * piece_seconds)
    pieces = [(s, min(s + per, N)) for s in range(0, N, per)]
    t0 = time.time()
    done_before = sum(b - a for i, (a, b) in enumerate(pieces)
                      if (parts / f"part_{i:05d}.done").exists())
    made = 0
    for i, (a, b) in enumerate(pieces):
        part, flag = parts / f"part_{i:05d}.mp4", parts / f"part_{i:05d}.done"
        if flag.exists():
            continue
        w = imageio.get_writer(str(part), fps=fps, codec="libx264", quality=8,
                               pixelformat="yuv420p", macro_block_size=1)
        try:
            for kf in range(a, b):
                w.append_data(np.asarray(dive.frame(kf, N, px=px, depth=depths[kf])))
                made += 1
                if progress:
                    progress(done_before + made, N, time.time() - t0, made)
        finally:
            w.close()
        flag.touch()
    lst = parts / "list.txt"
    lst.write_text("".join(f"file '{(parts / f'part_{i:05d}.mp4').resolve()}'\n"
                           for i in range(len(pieces))))
    subprocess.run([imageio_ffmpeg.get_ffmpeg_exe(), "-y", "-loglevel", "error", "-f", "concat",
                    "-safe", "0", "-i", str(lst), "-c", "copy", str(path)], check=True)
    shutil.rmtree(parts, ignore_errors=True)
    return path, time.time() - t0

# warm up (compiles the kernels once) and report
_t0 = time.time()
compute(0.0, 8, 64)
_r = RefOrbit(hp(-0.75 + 0.1j, 128), 128)
compute(40.0, 8, 64, ref=_r)
compute(4000.0, 8, 64, ref=_r)
print(f"Engine ready · {device_name()} · {time.time() - _t0:.1f} s to compile · "
      f"{len(DESTINATIONS)} destinations · films go to {MOVIE_DIR}/ · run cell 3 for the Studio.")

Engine ready · GPU · NVIDIA A100-SXM4-80GB · 3.1 s to compile · 14 destinations · films go to movies/ · run cell 3 for the Studio.


In [4]:
# 3 — The Studio: choose a destination, look around, preview, film
#
#   Destination menu   spirals, branch points, minibrot cascades, deep minibrots — or Custom
#   Viewer             left-click: zoom in toward the click (×2 … ×10,000, see "per click")
#                      right-click or shift-click: zoom out
#   🎯 Snap            (shallow views) the exactly self-repeating points nearest your click
#   🔭 Find minibrot   (any depth) the minibrot hiding in the view, to thousands of digits
#   👁 Preview         12 frames from along the film + a time estimate
#   🎬 Render film     renders in 10-second pieces; run it again with the same settings
#                      after a disconnect and it carries on where it stopped

import base64
import html
import io
import math
import traceback

import numpy as np
import ipywidgets as W
from IPython.display import display, Video, Javascript, clear_output
from PIL import ImageDraw

try:
    from google.colab import files as _colab_files, output as _colab_output
    IN_COLAB = True
except ImportError:
    _colab_files = _colab_output = None
    IN_COLAB = False

CUSTOM = "✎ Custom — explore, then 🎯 Snap or 🔭 Find minibrot"

# The viewer is a plain <img> inside a standard HTML widget, so it shows up in any
# notebook (VS Code, Jupyter, Colab) without add-on widget code. Clicks reach Python:
#   elsewhere: the img's click handler types "x,y,button,shift,time" into a hidden
#              standard Text widget, whose change Python observes;
#   on Colab:  a page-level listener calls Colab's own JavaScript→Python callback.
_ONCLICK = html.escape(
    "if (window.google && google.colab) return false;"
    "var r = this.getBoundingClientRect();"
    "var x = (event.clientX - r.left) / r.width * 2 - 1;"
    "var y = 1 - (event.clientY - r.top) / r.height * 2;"
    "var b = event.type == 'contextmenu' ? 3 : 1;"
    "var t = this.closest('.mandel-studio');"
    "var i = t && t.querySelector('.mandel-sink input');"
    "if (i) { i.value = [x, y, b, event.shiftKey ? 1 : 0, Date.now()].join(',');"
    "         i.dispatchEvent(new Event('input', {bubbles: true})); }"
    "return false;", quote=True)

_CLICK_JS = """
(() => {
  if (window.__mandelClicks) return;
  window.__mandelClicks = true;
  const send = (e, button) => {
    const el = e.target.closest && e.target.closest('.mandel-viewer');
    if (!el) return false;
    const r = el.getBoundingClientRect();
    const x = (e.clientX - r.left) / r.width * 2 - 1;
    const y = 1 - (e.clientY - r.top) / r.height * 2;
    google.colab.kernel.invokeFunction('mandel.click', [x, y, button, e.shiftKey], {});
    return true;
  };
  document.addEventListener('click', e => { send(e, 1); }, true);
  document.addEventListener('contextmenu', e => { if (send(e, 3)) e.preventDefault(); }, true);
})();
"""

def _png(img):
    buf = io.BytesIO()
    img.save(buf, "PNG")
    return buf.getvalue()

def _viewer_html(img):
    src = "data:image/png;base64," + base64.b64encode(_png(img)).decode()
    return (f'<img class="mandel-viewer" src="{src}" width="{img.width}" height="{img.height}" '
            f'style="display:block; cursor:crosshair" draggable="false" '
            f'onclick="{_ONCLICK}" oncontextmenu="{_ONCLICK}">')

def _mmss(s):
    s = int(round(s))
    return f"{s // 3600}:{s // 60 % 60:02d}:{s % 60:02d}" if s >= 3600 else f"{s // 60}:{s % 60:02d}"

class Studio:
    def __init__(self, view_px=480):
        self.view_px = view_px
        self.depth = 0.0                         # zoom ×2^depth
        self.center = hp(CENTER_HOME, 64)        # high-precision view center
        self.dest = None
        self.pick = None
        self.custom = None
        self.busy = False
        self._quiet = False
        self._build()
        self._on_dest({"new": self.dest_dd.value})

    # ------------------------------------------------------------ layout
    def _build(self):
        wide = W.Layout(width="780px")
        self.dest_dd = W.Dropdown(options=list(DESTINATIONS) + [CUSTOM], description="Destination",
                                  layout=wide, style={"description_width": "90px"})
        self.dest_dd.observe(self._on_dest, "value")
        self.info = W.HTML(layout=wide)

        self.view_html = W.HTML(layout=W.Layout(width=f"{self.view_px}px",
                                                height=f"{self.view_px}px"))
        self.view_out = self.view_html
        self.sink = W.Text(layout=W.Layout(display="none"))   # receives viewer clicks
        self.sink.add_class("mandel-sink")
        self.sink.observe(self._on_sink, "value")
        self.last_view = None                    # PIL image of the viewer

        def button(label, fn, tip=""):
            b = W.Button(description=label, tooltip=tip, layout=W.Layout(width="auto"))
            b.on_click(lambda _: self._guard(fn))
            return b
        self.btn_home = button("⌂ Whole set", self.go_home)
        self.btn_goto = button("◎ Go to destination", self.go_dest)
        self.btn_in = button("+ Zoom in", lambda: self.zoom(0j, self.per_click.value),
                             "Zoom in on the middle of the view")
        self.btn_out = button("− Zoom out", lambda: self.zoom(0j, 1.0 / self.per_click.value))
        self.per_click = W.Dropdown(options=[("×2", 2.0), ("×4", 4.0), ("×10", 10.0),
                                             ("×100", 100.0), ("×10,000", 1e4)], value=2.0,
                                    description="per click", layout=W.Layout(width="170px"),
                                    style={"description_width": "60px"})
        self.detail = W.Dropdown(options=[("×½", 0.5), ("×1", 1.0), ("×2", 2.0), ("×4", 4.0),
                                          ("×10", 10.0)], value=1.0, description="iterations",
                                 layout=W.Layout(width="170px"), style={"description_width": "70px"},
                                 tooltip="More iterations: crisper edges deep down, slower")
        self.detail.observe(lambda _: self._guard(self.render_view), "value")
        self.view_status = W.HTML(layout=W.Layout(width="490px"))

        self.julia_img = W.Image(format="png", layout=W.Layout(width="240px", height="240px"))
        self.julia_cap = W.HTML(layout=W.Layout(width="260px"))
        self.btn_snap = button("🎯 Snap to nearest repeating point", self.snap,
                               "Shallow views (zoom < ×10^12): uses your last click")
        self.btn_mini = button("🔭 Find the minibrot in this view", self.find_minibrot,
                               "Any depth: Newton's method to as many digits as needed")
        for b in (self.btn_snap, self.btn_mini):
            b.layout.width = "260px"
        self.cand_dd = W.Dropdown(options=[], layout=W.Layout(width="260px"), disabled=True)
        self.cand_dd.observe(self._on_cand, "value")
        self.pick_html = W.HTML(layout=W.Layout(width="260px"))

        self.seconds = W.IntSlider(value=60, min=10, max=1800, step=10, description="Length (s)",
                                   continuous_update=False)
        self.fps = W.Dropdown(options=[24, 30, 60], value=30, description="fps",
                              layout=W.Layout(width="140px"))
        self.size = W.Dropdown(options=[("480 px", 480), ("600 px", 600), ("720 px", 720),
                                        ("900 px", 900), ("1080 px", 1080)], value=600,
                               description="Size", layout=W.Layout(width="170px"))
        self.decades = W.FloatSlider(value=100, min=1, max=1000, step=1, description="Depth 10^",
                                     readout_format=",.0f", continuous_update=False)
        self.cmap = W.Dropdown(options=["magma", "inferno", "twilight_shifted", "viridis",
                                        "cividis", "hot"], value="magma", description="Colors",
                               layout=W.Layout(width="230px"))
        self.cmap.observe(lambda _: self._guard(self.render_view), "value")
        self.corot = W.Checkbox(value=True, description="Turn the camera with the spiral",
                                indent=False)
        self.strobe = W.Checkbox(value=True, indent=False,
                                 description="Strobe: jump whole repeats per frame (smooth at any speed)")
        self.embed = W.Checkbox(value=False, description="Show the film here (embeds the mp4)",
                                indent=False)
        self.speed = W.HTML()
        for w in (self.seconds, self.fps, self.decades, self.strobe, self.corot):
            w.observe(lambda _: self._update_speed(), "value")

        self.btn_prev = button("👁 Preview", self.preview)
        self.btn_film = button("🎬 Render film", self.film)
        self.btn_film.button_style = "warning"
        self.btn_dl = button("⬇ Download", self.download)
        self.btn_dl.layout.display = "none"
        self.prog = W.FloatProgress(value=0, min=0, max=100, layout=W.Layout(width="260px"))
        self.stat = W.HTML()
        self.prev_img = W.Image(format="png", layout=W.Layout(max_width="780px", display="none"))
        self.vid_out = W.Output()
        self.last_path = None

        viewer = W.VBox([self.view_out,
                         W.HBox([self.btn_home, self.btn_goto, self.btn_in, self.btn_out]),
                         W.HBox([self.per_click, self.detail]), self.view_status])
        side = W.VBox([W.HTML("<b>Julia twin</b>"), self.julia_img, self.julia_cap,
                       W.HTML("<hr style='margin:6px 0'>"), self.btn_snap, self.cand_dd,
                       self.btn_mini, self.pick_html], layout=W.Layout(margin="0 0 0 14px"))
        movie = W.VBox([W.HTML("<b>Film</b>"),
                        W.HBox([self.seconds, self.fps, self.size]),
                        W.HBox([self.decades, self.cmap]),
                        W.HBox([self.corot, self.strobe]), self.embed, self.speed,
                        W.HBox([self.btn_prev, self.btn_film, self.prog, self.btn_dl]), self.stat])
        self.ui = W.VBox([W.HTML(f"<small style='color:#888'>{device_name()}</small>"),
                          self.dest_dd, self.info, W.HBox([viewer, side]),
                          W.HTML("<hr style='margin:8px 0'>"), movie, self.prev_img, self.vid_out,
                          self.sink])
        self.ui.add_class("mandel-studio")
        self.buttons = [self.btn_home, self.btn_goto, self.btn_in, self.btn_out, self.btn_snap,
                        self.btn_mini, self.btn_prev, self.btn_film, self.btn_dl]

    # ------------------------------------------------------------ helpers
    def _run(self, fn):
        if self.busy:
            fn()
        else:
            self._guard(fn)

    def _guard(self, fn):
        if self.busy:
            return
        self.busy = True
        for b in self.buttons:
            b.disabled = True
        try:
            fn()
        except Exception:
            self.stat.value = f"<pre style='color:#c33'>{traceback.format_exc()}</pre>"
        finally:
            self.busy = False
            for b in self.buttons:
                b.disabled = False
            self.btn_film.disabled = self.btn_prev.disabled = self.dest is None

    @property
    def bits(self):
        return bits_for(self.depth + 8)

    @property
    def half(self):
        """Half the view width, as a high-precision number."""
        with precision(self.bits):
            return mpfr(SPAN_HOME / 2) * mpfr(2) ** mpfr(-self.depth)

    def _rel(self, z):
        """High-precision point -> view coordinates (-1..1), or None if far outside."""
        with precision(self.bits):
            w = (z - self.center) / self.half
        if abs(w.real) > 1e6 or abs(w.imag) > 1e6:
            return None
        return complex(w)

    def _set_dest(self, dest):
        self.dest = dest
        if dest is None:
            self.info.value = ("<i>Custom: zoom in by left-clicking where you like. Near the "
                               "surface use 🎯 Snap; at any depth use 🔭 Find minibrot.</i>")
            self.julia_img.value = b""
            self.julia_img.layout.visibility = "hidden"
            self.julia_cap.value = ""
            self.btn_film.disabled = self.btn_prev.disabled = True
            self._update_speed()
            return
        self.info.value = (f"<div style='line-height:1.35'>{dest.blurb}<br>"
                           f"<small style='color:#888'>{dest.summary()}</small></div>")
        top = dest.max_halvings * LOG10_2
        self.decades.max = max(top, 1.0)
        self.decades.step = 1.0 if top > 30 else 0.5
        if dest.kind == "misiurewicz":
            self.decades.value = min(top, 1000.0 if GPU else 100.0)
        else:
            self.decades.value = top
        self.corot.disabled = not dest.can_turn
        self.corot.description = ("Turn so the minibrot ends upright" if dest.kind == "minibrot"
                                  else "Turn the camera with the spiral")
        if not dest.can_turn:
            self.corot.value = False
        self.strobe.disabled = dest.kind != "misiurewicz"
        self.btn_film.disabled = self.btn_prev.disabled = False
        self.julia_img.value = _png(Image.fromarray(colorize(julia(dest.c, 240), "magma")))
        self.julia_img.layout.visibility = "visible"
        self.julia_cap.value = ("<small>Julia set of c*. Zoomed far enough in at c*, the "
                                "Mandelbrot set looks like this set near c* (Tan Lei).</small>"
                                if dest.kind == "misiurewicz" else
                                "<small>Julia set of c* — the dynamics behind the cascade.</small>"
                                if dest.kind == "renorm" else
                                "<small>Julia set of the minibrot's nucleus (a period-"
                                f"{dest.p:,d} 'rabbit' of sorts).</small>")
        self._update_speed()

    def _update_speed(self):
        if self.dest is None:
            self.speed.value = ""
            return
        N = self.seconds.value * self.fps.value
        H = self.decades.value / LOG10_2
        txt = (f"<small>{N:,d} frames · zooming ×10^{self.decades.value / self.seconds.value:.2f} "
               f"per second")
        if self.dest.kind == "misiurewicz":
            per = H / self.dest.log2_scale / max(N - 1, 1)
            if per > 0.5:
                txt += (f" · <b>{per:,.1f} repeats per frame</b> — "
                        + ("strobe on: the film looks like one slow, steady repeat while the "
                           "counter races" if self.strobe.value else
                           "<span style='color:#c63'>too fast to follow: turn strobe on or make "
                           "the film longer</span>"))
        elif self.decades.value / self.seconds.value > 1.0:
            txt += " · <span style='color:#c63'>fast — a longer film is easier to watch</span>"
        self.speed.value = txt + "</small>"

    # ------------------------------------------------------------ viewer
    def _view_ref(self):
        """Reference orbit for the viewer: the destination itself when it is in
           view (exact, and fast inside minibrots), else the view center."""
        d, bits = self.dest, self.bits
        if d is not None:
            z = d.ref_hp(bits)
            w = self._rel(z)
            if w is not None and abs(w) < 8:
                cyc = (d.k, d.p) if d.kind == "misiurewicz" else (-1, d.p) if d.kind == "minibrot" else (0, 0)
                return get_ref(z, bits), cyc
        with precision(bits):
            c = mpc(self.center)
        return get_ref(c, bits), (0, 0)

    def _iters(self):
        n = auto_iter(self.depth)
        d = self.dest
        if d is not None and d.kind == "minibrot":
            w = self._rel(d.ref_hp(self.bits))
            if w is not None and abs(w) < 8:
                n = max(n, Dive(d).max_iter(min(self.depth, d.max_halvings)))
        return int(n * self.detail.value)

    def render_view(self):
        t0 = time.time()
        px, mi = self.view_px, self._iters()
        if self.depth < DIRECT_HALVINGS:
            mu = compute(self.depth, px, mi, complex(self.center))
            how = "float64"
        else:
            ref, cyc = self._view_ref()
            m, E = grid(self.depth, px)
            with precision(self.bits):
                off = complex((self.center - ref.c) * mpfr(2) ** (-E))
            mu = compute(self.depth, px, mi, ref=ref, off=off, cycle=cyc)
            how = f"perturbation · {ndigits(ref.c):,d} digits"
        rgb = colorize(mu, self.cmap.value)
        ring = plus = None
        if self.dest is not None:
            w = self._rel(self.dest.ref_hp(self.bits))
            if w is not None and abs(w.real) < 1 and abs(w.imag) < 1:
                ring = w
        if self.pick is not None:
            w = self._rel(self.pick)
            if w is not None and abs(w.real) < 1 and abs(w.imag) < 1:
                plus = w
        img = Image.fromarray(rgb)
        dr = ImageDraw.Draw(img)
        to_px = lambda w: ((w.real + 1) / 2 * px, (1 - w.imag) / 2 * px)
        if ring is not None:                     # gold ring: the destination
            x, y = to_px(ring)
            r = px / 60
            dr.ellipse((x - r, y - r, x + r, y + r), outline=(255, 205, 40), width=3)
        if plus is not None:                     # white +: your last click
            x, y = to_px(plus)
            r = px / 70
            dr.line((x - r, y, x + r, y), fill=(255, 255, 255), width=2)
            dr.line((x, y - r, x, y + r), fill=(255, 255, 255), width=2)
        self.last_view = img
        self.view_html.value = _viewer_html(img)
        self.view_status.value = (f"<small>center {short(self.center, 16)}<br>"
                                  f"zoom {zoom_text(self.depth)} · {mi:,d} iterations · {how} · "
                                  f"{time.time() - t0:.1f} s</small>")

    def zoom(self, rel, factor):
        """Zoom by `factor` keeping the view point `rel` (-1..1) fixed."""
        new_depth = max(self.depth + math.log2(factor), -0.5)
        f = 2.0 ** (self.depth - new_depth)          # new span / old span
        bits = bits_for(new_depth + 8)
        with precision(bits):
            self.center = self.center + mpc(rel) * self.half * (1 - f)
        self.depth = new_depth
        self.render_view()

    def go_home(self):
        self.center, self.depth = hp(CENTER_HOME, 64), 0.0
        self.render_view()

    def _dest_view(self, depth):
        dv = Dive(self.dest, halvings=self.dest.max_halvings)
        m, E = grid(depth, self.view_px)
        center, off = dv._center(depth, E)
        if depth < DIRECT_HALVINGS:
            return hp(center, bits_for(depth + 8))
        bits = bits_for(depth + 8)
        with precision(bits):
            return self.dest.ref_hp(bits) + mpc(off) * mpfr(2) ** E

    def go_dest(self):
        d = self.dest
        if d is None:
            return
        if d.kind == "minibrot":
            depth = d.max_halvings
        else:
            depth = min(math.ceil(10.0 / d.log2_scale) * d.log2_scale, d.max_halvings)
        self.center, self.depth = self._dest_view(depth), depth
        self.render_view()

    def _on_sink(self, change):
        """A viewer click typed into the hidden Text widget: "x,y,button,shift,time"."""
        try:
            x, y, b, s = change["new"].split(",")[:4]
            self.click(float(x), float(y), int(float(b)), s.strip() == "1")
        except ValueError:
            pass

    def click(self, x, y, button=1, shift=False):
        """A click at view coordinates (x, y) in -1..1: left zooms in, right/shift out."""
        if self.busy:
            return
        rel = complex(max(-1.0, min(1.0, float(x))), max(-1.0, min(1.0, float(y))))
        out = int(button) == 3 or bool(shift)
        if not out and int(button) == 1:
            with precision(self.bits):
                self.pick = self.center + mpc(rel) * self.half
            self.pick_html.value = f"<small>last click: {short(self.pick, 12)}</small>"
        k = self.per_click.value
        self._guard(lambda: self.zoom(rel, 1.0 / k if out else k))

    # ------------------------------------------------------------ choosing
    def _on_dest(self, change):
        if self._quiet:
            return
        name = change["new"]
        if name == CUSTOM:
            self._set_dest(self.custom)
            self.cand_dd.disabled = not self.cand_dd.options
            self._guard(self.render_view)
        else:
            self._set_dest(DESTINATIONS[name])
            self._guard(self.go_dest)

    def _make_custom(self, dest, note):
        self.custom = dest
        self._quiet = True
        self.dest_dd.value = CUSTOM
        self._quiet = False
        self._set_dest(dest)
        self.pick_html.value = note
        self._run(self.render_view)

    def snap(self):
        if self.depth > 40:
            self.pick_html.value = ("<small style='color:#c63'>Too deep for 🎯 (it works in float64, "
                                    "to ×10^12). Use 🔭 Find minibrot here.</small>")
            return
        c0 = complex(self.pick if self.pick is not None else self.center)
        self.pick_html.value = "<small>searching preperiods 2–24, periods 1–8 …</small>"
        span = SPAN_HOME * 2.0 ** -self.depth
        cands = snap_misiurewicz(c0, radius=span) or snap_misiurewicz(c0)
        if not cands:
            self.pick_html.value = ("<small style='color:#c33'>No self-repeating point found "
                                    "there — click right on the fringe and try again.</small>")
            return
        opts = []
        for i, (c, k, p, lam) in enumerate(cands):
            slow = " · very slow" if abs(lam) < 1.05 else ""
            opts.append((f"{i + 1}. ×{abs(lam):.3g} per repeat, {math.degrees(np.angle(lam)):+.0f}° · "
                         f"k={k}, p={p}{slow}", i))
        self._cands = cands
        self._quiet = True
        self.dest_dd.value = CUSTOM
        self.cand_dd.options = opts
        self.cand_dd.disabled = False
        self.cand_dd.value = None
        self._quiet = False
        self.cand_dd.value = 0           # -> _on_cand
        self.pick_html.value = (f"<small>{len(cands)} found near your click; gold ring = chosen. "
                                f"'very slow' = barely repeats.</small>")

    def _on_cand(self, change):
        if self._quiet or change["new"] is None:
            return
        c, k, p, lam = self._cands[change["new"]]
        d = Dest.misiurewicz(f"Your point (preperiod {k}, period {p})", c,
                             f"Snapped from your click to c* = {c.real:+.12f} {c.imag:+.12f}i.")
        self._make_custom(d, self.pick_html.value)

    def find_minibrot(self):
        def status(s):
            self.pick_html.value = f"<small>{s}</small>"
        status("looking for the lowest-period minibrot in view …")
        t0 = time.time()
        r = minibrot_near(self.center, self.depth, status=status)
        if r is None:
            status("<span style='color:#c33'>No minibrot found here (period > 400,000?). "
                   "Zoom in a little and try again.</span>")
            return
        nuc, p, size = r
        d = Dest.minibrot(f"Minibrot of period {p:,d}", nuc, p, size,
                          f"Found from your view: a period-{p:,d} minibrot, its nucleus known to "
                          f"{ndigits(nuc):,d} digits. The film zooms from the whole set down to it.")
        self._make_custom(d, f"<small>period {p:,d} · 10^-{d.max_halvings * LOG10_2:,.0f} across · "
                             f"tilted {d.angle:+.0f}° · {time.time() - t0:.1f} s. "
                             f"◎ Go to destination to see it.</small>")

    # ------------------------------------------------------------ filming
    def _dive(self):
        return Dive(self.dest, halvings=self.decades.value / LOG10_2, cmap=self.cmap.value,
                    corotate=self.corot.value, strobe=self.strobe.value,
                    detail=self.detail.value, frames_per_repeat=3 * self.fps.value)

    def preview(self):
        N = self.seconds.value * self.fps.value
        self.stat.value = "computing the reference orbit and 12 preview frames …"
        sheet, per_frame = contact_sheet(self._dive(), N, px=self.size.value)
        self.prev_img.value = _png(sheet)
        self.prev_img.layout.display = ""
        self.stat.value = (f"Preview ready · {N:,d} frames · rough estimate {_mmss(per_frame * N)} "
                           f"(h:mm:ss) on this {('GPU' if GPU else 'CPU')} — deep frames are the slow ones")

    def film(self):
        N = self.seconds.value * self.fps.value
        dive = self._dive()
        path = film_path(dive, N, self.fps.value, self.size.value)
        self.btn_dl.layout.display = "none"
        if path.exists():
            self.stat.value = f"✅ already rendered: <code>{path}</code>"
        else:
            parts = path.parent / (path.stem + "_parts")
            resume = parts.is_dir() and any(parts.glob("*.done"))
            self.prog.value = 0
            self.prog.bar_style = ""
            def progress(k, n, el, made):
                if made % 5 == 0 or k == n:
                    self.prog.value = 100.0 * k / n
                    self.stat.value = (f"frame {k:,d}/{n:,d} · {_mmss(el)} elapsed · "
                                       f"~{_mmss(el / made * (n - k))} left at this pace")
            self.stat.value = (("resuming " if resume else "rendering ")
                               + f"{N:,d} frames → {path} …")
            path, el = write_movie(dive, path, N, fps=self.fps.value, px=self.size.value,
                                   progress=progress)
            self.prog.bar_style = "success"
            self.stat.value = (f"✅ saved <code>{path}</code> · {path.stat().st_size / 1e6:.1f} MB · "
                               f"{_mmss(el)}")
        self.last_path = path
        if _colab_files is not None:
            self.btn_dl.layout.display = ""
        with self.vid_out:
            clear_output()
            if self.embed.value:
                display(Video(str(path), embed=True, width=640))

    def download(self):
        if self.last_path is not None and _colab_files is not None:
            _colab_files.download(str(self.last_path))

if "DESTINATIONS" not in globals() or "minibrot_near" not in globals():
    print("Run cell 2 (the engine) first.")
else:
    studio = Studio()
    display(studio.ui)
    if _colab_output is not None:
        _colab_output.register_callback("mandel.click", studio.click)
        display(Javascript(_CLICK_JS))

<IPython.core.display.Javascript object>

```markdown
### Why does rendering slow down exponentially as we zoom deeper?

This behavior is a direct consequence of the mathematics of the Mandelbrot set and how the **Perturbation Engine** calculates pixel escapes at extreme depths.

#### 1. The Iteration Scaling
As you zoom deeper into a structure (especially toward a **minibrot** or a complex boundary spiral):
* **The Boundary Complexity Increases:** Near the center of the set, orbits take much longer to either escape to infinity or resolve to their attracting cycle.
* **Mathematical Scaling:** The number of iterations required to resolve the details at a zoom depth of $10^{-D}$ scales exponentially. In the engine's `max_iter` scheduling:
  $$\text{iterations} \propto D^3$$
  For example, a shallow view might need only $1,000$ iterations per pixel, but a frame at $10^{-1000}$ depth can require upwards of **$3,000,000$ iterations per pixel** to correctly resolve the tiny boundary features.

#### 2. GPU Thread Divergence
At shallow depths, neighboring pixels behave very similarly (either they all escape quickly or all stay inside).
* On a GPU, execution is grouped in "warps" of 32 parallel threads.
* At extreme depths, some pixels inside a warp escape in $100$ steps, while others require $3,000,000$ steps.
* Because of GPU hardware architecture, **all 32 threads in a warp must wait for the slowest pixel to finish iterating** before any of those threads can accept new work. This "thread divergence" dramatically reduces effective GPU parallel efficiency.

#### 3. Increased Precision Overhead (Reference Orbit Calculation)
Before rendering a deep frame, a single high-precision "reference orbit" must be calculated on the CPU using `gmpy2` with up to $3,400$ bits of precision. While calculating one orbit is generally fast, as the length of this orbit grows to millions of steps, the CPU overhead of preparing the reference data before handing it over to the GPU starts to add up.
```

In [ ]:
def calculate_render_bandwidth(px, max_iter, actual_render_time, gpu_name="NVIDIA A100"):
    """
    Calculates the memory bandwidth usage for the deep-zoom Mandelbrot engine.

    Parameters:
    - px: Width/height of the frame in pixels (e.g., 600)
    - max_iter: Maximum or average iterations performed per pixel
    - actual_render_time: The execution time of the render kernel in seconds
    - gpu_name: Name of the hardware for reference comparison
    """
    total_pixels = px * px

    # Each active thread reads 1 real and 1 imaginary value from the reference orbit arrays
    # (each float64 = 8 bytes) per iteration step.
    bytes_read_per_iteration = 16  # 8 bytes for Zr[n] + 8 bytes for Zi[n]

    # Estimated total bytes read during kernel execution
    total_bytes_transferred = total_pixels * max_iter * bytes_read_per_iteration
    total_gigabytes = total_bytes_transferred / (1024 ** 3)

    # Bandwidth = Gigabytes / rendering time
    effective_bandwidth_gb_s = total_gigabytes / actual_render_time

    print(f"--- Memory Bandwidth Calculation for {gpu_name} ---")
    print(f"Resolution: {px} x {px} ({total_pixels:,} total pixels)")
    print(f"Iterations per pixel: {max_iter:,}")
    print(f"Total Data Transferred: {total_gigabytes:.2f} GB")
    print(f"Render Time: {actual_render_time:.4f} seconds")
    print(f"Effective Bandwidth: {effective_bandwidth_gb_s:.2f} GB/s")

    # Reference comparisons
    if "A100" in gpu_name:
        peak_bw = 2039 # GB/s for A100
        utilization = (effective_bandwidth_gb_s / peak_bw) * 100
        print(f"A100 Peak Theoretical Bandwidth: 2,039 GB/s")
        print(f"Memory Bus Saturation (utilization): {utilization:.2f}%")

# Example calculation:
# Rendering a 600x600 frame with 1,000,000 iterations that takes 2.5 seconds on an A100
calculate_render_bandwidth(px=600, max_iter=1000000, actual_render_time=2.5)